In [1]:
import torch
import torch.nn as nn
import math


## Input Embedding


In [2]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


"""
 V important: 
    
    we did not use default nn.Embedding initialiazation and multiply it by sqrt(d_model)!!
    The authors initialized the embedding values with mean = 0 and var = 1/dmodel 
    (check in original tensor2tensor library source code by Google) 
    because they are going to use the same embedding values for the pre-softmax 
    transformation too. 
    So they multiplied the embedding values by sqrt(d_model) to scale up the variance to 1 
    (matching the scale of the P.E. (mean ~ 0, var ~ 0.5) values for addition). 
    Since the default nn.Embedding class already has a variance of 1, 
    we must re-initialize it with variance = 1/d_model to replicate the paper(we did it in build_transformer function(last cell of model.ipynb)). 
    Without re-initializing, the sqrt(d_model) multiplication would explode the var to d_model(512).
    
"""


'\n V important: \n    \n    we did not use default nn.Embedding initialiazation and multiply it by sqrt(d_model)!!\n    The authors initialized the embedding values with mean = 0 and var = 1/dmodel \n    (check in original tensor2tensor library source code by Google) \n    because they are going to use the same embedding values for the pre-softmax \n    transformation too. \n    So they multiplied the embedding values by sqrt(d_model) to scale up the variance to 1 \n    (matching the scale of the P.E. (mean ~ 0, var ~ 0.5) values for addition). \n    Since the default nn.Embedding class already has a variance of 1, \n    we must re-initialize it with variance = 1/d_model to replicate the paper(we did it in build_transformer function(last cell of model.ipynb)). \n    Without re-initializing, the sqrt(d_model) multiplication would explode the var to d_model(512).\n    \n'

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a positional encoding matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # create a vector of shape (seq_len, 1) containing the position indices
        position = torch.arange(
            0, seq_len, dtype=torch.float
        ).unsqueeze(
            1
        )  # this will output a tensor of shape (seq_len, 1) containing the position indices from 0 to seq_len-1
        print(position.shape)
        # Compute 1/10000^(2i/d_model) using exp(log(...)) for numerical stability with float32:
        # e^(i * (-ln(10000) / d_model)) = 1 / 10000^(i / d_model) where i = 0, 2, 4, ..., d_model-2
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        # Apply sine to even indices (0, 2, 4, ...) and cosine to odd indices (1, 3, 5, ...) of the positional encoding matrix
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Add a batch dimension: shape changes from (seq_len, d_model) to (1, seq_len, d_model), So the same positional encoding is used for every sentence/example in the batch.
        pe = pe.unsqueeze(0)
        print(pe.shape)
        # Register 'pe' as a buffer instead of a parameter. Buffers don't require gradients
        # (won't be updated by the optimizer), but they are saved in the model's state_dict
        # and automatically moved to the correct device (CPU/GPU) along with the model.
        self.register_buffer("pe", pe)

    def forward(self, x):
        # Add the positional encoding to the input embeddings and apply dropout
        x = (
            x + (self.pe[:, : x.shape[1], :])
        )  # Ensure positional encodings are not updated during training
        # or x = x + self.pe[0, :seq_len, 511]?
        return self.dropout(x)

In [4]:
class LayerNorm(nn.Module):
    def __init__(self, eps: float = 10**-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mean = x.mean(
            -1, keepdim=True
        )  # mean of of last dimension (d_model) of x which has shape (batch_size, seq_len, d_model) and keepdim=True to maintain the same dimention as mean() reduces the dimention so without it the shape of mean would be (batch_size, seq_len) and we want it to be (batch_size, seq_len, 1) so that we can broadcast it to the shape of x when we subtract it from x
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        # this first linear layer will increase the dimention to d_ff
        self.linear1 = nn.Linear(
            d_model, d_ff
        )  # bias of shape (d_ff,) is True by default so we don't need to specify it
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(
            d_ff, d_model
        )  # returns a tensor of shape (batch_size, seq_len, d_model), bias of shape (d_model,) is true by default

    def forward(self, x):
        x = self.linear1(
            x
        )  # (batch_size, seq_len, d_model) -> (batch_size, seq_len, d_ff)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [6]:
class MultiheadAttnBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model should be divisible by h"

        self.d_k = d_model // h
        self.d_v = d_model // h
        """
        the paper mentions the dimension of W_q is d_model x d_k (dimention or W_q for each head),
        but we are using d_model x d_model for computation efficiency(much faster than 8 sequential (d_model x d_k) multiplies (see the cell commented cell below this one for the sequential implementation)),
        which we will split into h heads after muliplication with Q, K, and V.  The same goes for W_k and W_v 
        """
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(h * self.d_v, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(Query, Key, Value, mask, dropout: nn.Dropout):
        d_k = Query.size(-1)  # dimension of the key vectors
        attention_scores = (
            (Query @ Key.transpose(-2, -1)) / math.sqrt(d_k)
        )  # (batch_size, h, seq_len, d_k) of queries and keys  -> (batch_size, h, seq_len, seq_len) of score_matrix
        if mask is not None:
            attention_scores = attention_scores.masked_fill(
                mask == 0, -1e9
            )  # set masked positions to very large negative value so that softmax will make them ~0
            # we don't usefloat('-inf') because it sometimes cause PyTorch's softmax to output NaN errors

        attention_weights = attention_scores.softmax(
            dim=-1
        )  # (batch_size, h, seq_len, seq_len)
        """
        softmax is applied horizontally across the columns(dim = -1).For a single row, the softmax converts the raw scores across all the columns 
        into percentages. The values in each row will sum up to 1.0 (or 100%). 
        In plain English: "For this specific Query word (row), 
        what percentage of its attention should it pay to each of the Key words (columns) 
        in the sentence?
        """
        if dropout is not None:
            attention_weights = dropout(attention_weights)
        return (
            attention_weights @ Value,
            attention_weights,
        )  # (batch_size, h, seq_len, d_v), (batch_size, h, seq_len, seq_len)

    def forward(self, Q, K, V, mask):
        query = self.W_q(Q)  # (batch_size, seq_len, d_model)
        key = self.W_k(K)  # (batch_size, seq_len, d_model)
        value = self.W_v(V)  # (batch_size, seq_len, d_model)

        # split into h heads
        Q_i = query.view(query.size(0), query.size(1), self.h, self.d_k).transpose(
            1, 2
        )  # (batch_size, seq_len, d_model) -> (batch_size, h, seq_len, d_k)
        """
        we transposed the 2nd and 3rd dimensions from (batch_size, seq_len, h, d_k) to (batch_size, h, seq_len, d_k)
        because attention operates on the last two dimensions: (seq_len, d_k)
        PyTorch's @ treats all leading dimensions as batch dimensions and only multiplies the last two. 
        By having (batch, h, ...), both batch and h act as independent batch dimensions so all 8 heads are computed in parallel in a single @ operation.
        """
        K_i = key.view(key.size(0), key.size(1), self.h, self.d_k).transpose(
            1, 2
        )  # (batch_size, seq_len, d_model) -> (batch_size, h, seq_len, d_k)
        V_i = value.view(value.size(0), value.size(1), self.h, self.d_v).transpose(
            1, 2
        )  # (batch_size, seq_len, d_model) -> (batch_size, h, seq_len, d_v)

        x, self.attention_weights_for_visualization = self.attention(
            Q_i, K_i, V_i, mask, self.dropout
        )  # (batch_size, h, seq_len, d_v), (batch_size, h, seq_len, seq_len)

        # (batch_size, h, seq_len, d_v) -> (batch_size, seq_len, h, d_v)
        x = x.transpose(1, 2)  # to concat all the heads together
        x = x.contiguous().view(
            x.size(0), x.size(1), self.h * self.d_v
        )  # (batch_size, seq_len, h * d_v) of x  and (batch_size, seq_len, h * d_v)

        return self.W_o(
            x
        )  # x.W_o #(batch_size, seq_len, h * d_v) of x and (batch_size, h * d_v, d_model) of W_o -> (batch_size, seq_len, d_model) of output

In [7]:
# MultiheadAttnBlock implementation using sequential implementation of attention for each head (slower than the optimized version above) matching paper's dimensions of W_q, W_k, and W_v (d_model x d_k) for each head. This is commented out because it is slower than the optimized version above.

# class MultiheadAttnBlock(nn.Module):
#     def __init__(self, d_model: int, h: int, dropout: float) -> None:
#         super().__init__()
#         self.d_model = d_model
#         self.h = h
#         assert d_model % h == 0, "d_model should be divisible by h"

#         self.d_k = d_model // h  # dk = dv = d_model / h = 64
#         self.d_v = d_model // h

#         #paper's Wi_q (d_model x d_k) per head
#         self.W_q = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(h)])
#         self.W_k = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(h)])
#         self.W_v = nn.ModuleList([nn.Linear(d_model, self.d_v) for _ in range(h)])

#         #paper's W_o ∈ (h·d_v × d_model)
#         self.W_o = nn.Linear(h * self.d_v, d_model)
#         self.dropout = nn.Dropout(dropout)

#     @staticmethod
#     def attention(query, key, value, mask=None, dropout=None):
#         d_k = query.size(-1) # dimension of the key vectors
#         scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
#         if mask is not None:
#             scores = scores.masked_fill(mask == 0, -1e9)
#         attn_weights = torch.softmax(scores, dim=-1)
#         if dropout is not None:
#             attn_weights = dropout(attn_weights)
#         return attn_weights @ value, attn_weights

#     def forward(self, Q, K, V, mask):
#         heads = []
#         for i in range(self.h):
#             # Paper: head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)

#             Q_i = self.W_q[i](Q)  # (batch_size, seq_len, d_k)
#             K_i = self.W_k[i](K)  # (batch_size, seq_len, d_k)
#             V_i = self.W_v[i](V)  # (batch_size, seq_len, d_v)

#             # (batch_size, seq_len, d_v)
#             head_i, self.attention_weights_for_visualization = self.attention(Q_i, K_i, V_i, mask, self.dropout)
#             heads.append(head_i)
#         # Paper: Concat(head_1, ..., head_h)
#         # We call the concatenated output `x` to match the optimized version's flow
#         x = torch.cat(heads, dim=-1)  # (batch_size, seq_len, h * d_v)
#         # Paper: Concat(...) W^O
#         return self.W_o(x)  # (batch_size, seq_len, d_model)

In [8]:
class ResidualConnection(nn.Module):
    def __init__(self, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.Norm = LayerNorm()

    def forward(self, x, sublayer):
        # x is input from the previous layer, sublayer is the function to be applied (MultiheadAttnBlock or FeedForward)

        """
        in the paper this is the pattern of LN:
        self.Norm(x + self.dropout(sublayer(x)))
        this is called as Post-LN and it is very hard to train
        as it suffers from exploding gradients early in training,
        which is why the paper required a complex "learning rate warmup" schedule.

        Placing the LayerNorm inside the residual block (Pre-LN), as we did below,
        makes the gradients flow much more smoothly. Almost all modern Transformer models (like GPT-2/3, LLaMA, PaLM, Gemini etc)
        uses the Pre-LN configuration use the Pre-LN

        """

        return x + self.dropout(
            sublayer(self.Norm(x))
        )  # apply norn before the sublayer and add the residual connection

## Encoder


In [9]:
class Encoder(nn.Module):
    def __init__(
        self,
        self_attn_block: MultiheadAttnBlock,
        feed_forward: FeedForward,
        dropout: float,
    ):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.residual1 = ResidualConnection(dropout)
        self.feed_forward = feed_forward
        self.residual2 = ResidualConnection(dropout)

    def forward(
        self, x, src_mask
    ):  # we apply mask in the self-attention layer in encoder to prevent attending to padding tokens
        x = self.residual1(
            x, lambda x: self.self_attn_block(x, x, x, src_mask)
        )  # self-attention
        x = self.residual2(x, self.feed_forward)  # feed-forward
        return x

In [10]:
class NEncoders(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNorm()

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

    """
    since we are using Pre-LN, we apply the final LayerNorm after all the encoder layers.
    so we are doing this: LayerNorm(x + Sublayer(LayerNorm(x))).
    Because the very last operation in paper's residual connection was a LayerNorm, 
    the output of the final encoder layer was already perfectly normalized. 
    therefore, the paper did not have a final LayerNorm at the end of last encoder block.
    """

## Decoder


In [11]:
class Decoder(nn.Module):
    def __init__(
        self,
        self_attn_block: MultiheadAttnBlock,
        cross_attn_block: MultiheadAttnBlock,
        feed_forward: FeedForward,
        dropout: float,
    ):
        super().__init__()
        self.self_attn_block = self_attn_block
        self.residual1 = ResidualConnection(dropout)
        self.cross_attn_block = cross_attn_block
        self.residual2 = ResidualConnection(dropout)
        self.feed_forward = feed_forward
        self.residual3 = ResidualConnection(dropout)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.self_attn_output = self.residual1(
            x, lambda x: self.self_attn_block(x, x, x, tgt_mask)
        )
        x = self.cross_attn_block_output = self.residual2(
            x,
            lambda x: self.cross_attn_block(
                x, encoder_output, encoder_output, src_mask
            ),
        )
        x = self.residual3(x, self.feed_forward)
        return x

In [12]:
class NDecoders(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNorm()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

In [13]:
class ProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.projection = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # (batch_size, seq_len, d_model) of x and (batch_size, d_model, vocab_size) of W_proj -> (batch_size, seq_len, vocab_size) of output
        return torch.log_softmax(self.projection(x), dim=-1)

In [15]:
class Transformer(nn.Module):
    def __init__(
        self,
        encoder: NEncoders,
        decoder: NDecoders,
        src_embedding: InputEmbedding,
        tgt_embedding: InputEmbedding,
        enc_pe: PositionalEncoding,
        dec_pe: PositionalEncoding,
        proj_layer: ProjectionLayer,
    ) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embedding = src_embedding
        self.tgt_embedding = tgt_embedding
        self.enc_pe = enc_pe
        self.dec_pe = dec_pe
        self.proj_layer = proj_layer

    def encode(self, src, src_mask):
        # src: (batch_size, src_seq_len)
        x = self.src_embedding(src)  # (batch_size, src_seq_len, d_model)
        x = self.enc_pe(x)  # (batch_size, src_seq_len, d_model)
        return self.encoder(x, src_mask)  # (batch_size, src_seq_len, d_model)

    def decode(self, tgt, encoder_output, src_mask, tgt_mask):
        # tgt: (batch_size, tgt_seq_len)
        x = self.tgt_embedding(tgt)  # (batch_size, tgt_seq_len, d_model)
        x = self.dec_pe(x)  # (batch_size, tgt_seq_len, d_model)
        return self.decoder(
            x, encoder_output, src_mask, tgt_mask
        )  # (batch_size, tgt_seq_len, d_model)

    def project(self, x):
        # x: (batch_size, tgt_seq_len, d_model)
        return self.proj_layer(x)  # (batch_size, tgt_seq_len, vocab_size)


def build_transformer(
    src_vocab_size: int,
    tgt_vocab_size: int,
    src_seq_len: int,
    tgt_seq_len: int,
    d_model: int = 512,
    d_ff: int = 2048,
    h: int = 8,
    N: int = 6,
    dropout: float = 0.1,
) -> Transformer:

    # create the embedding layers for source and target languages
    src_embed = InputEmbedding(d_model, src_vocab_size)
    tgt_embed = InputEmbedding(d_model, tgt_vocab_size)

    # create the pos encoding layers for source and target languages
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    # create the encoder block
    encoder_layers = []
    for _ in range(N):
        enc_self_attn_block = MultiheadAttnBlock(d_model, h, dropout)
        enc_feed_forward = FeedForward(d_model, d_ff, dropout)
        enc_block = Encoder(enc_self_attn_block, enc_feed_forward, dropout)
        encoder_layers.append(enc_block)

    # create the decoder block
    decoder_layers = []
    for _ in range(N):
        dec_self_attn_block = MultiheadAttnBlock(d_model, h, dropout)
        dec_cross_attn_block = MultiheadAttnBlock(d_model, h, dropout)
        dec_feed_forward = FeedForward(d_model, d_ff, dropout)
        dec_block = Decoder(
            dec_self_attn_block, dec_cross_attn_block, dec_feed_forward, dropout
        )
        decoder_layers.append(dec_block)

    # create encoder and decoder modules
    encoder = NEncoders(nn.ModuleList(encoder_layers))
    decoder = NDecoders(nn.ModuleList(decoder_layers))

    # create the projection layer
    proj_layer = ProjectionLayer(d_model, tgt_vocab_size)

    # create the transformer model
    transformer = Transformer(
        encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, proj_layer
    )

    # initialize the parameters of the transformer
    for name, p in transformer.named_parameters():
        if "embedding" in name:
            nn.init.normal_(p, mean=0, std=d_model**-0.5)
        elif p.dim() > 1:
            nn.init.xavier_uniform_(p)

    transformer.tgt_embedding.embedding.weight = (
        transformer.src_embedding.embedding.weight
    )
    transformer.proj_layer.projection.weight = (
        transformer.src_embedding.embedding.weight
    )

    """
    since the paper does not explicitly mentions about initialization,
    we mimic the initialization used in the original tensor2tensor library by Google 
    which is based on Xavier's initialization for all the parameters except for the 
    embedding layers which are initialized with mean=0 and var=1/d_model 
    to match the scale of positional encoding values (mean ~ 0, var ~ 0.5) for addition.
    """

    return transformer